# 🔬 exo2micro — Interactive Pipeline

This notebook provides an interactive interface for processing pre/post-stain
fluorescence microscopy images with **exo2micro 2.3**.

**What this pipeline does:**
1. Loads paired pre-stain and post-stain fluorescence images
2. Registers (aligns) the pre-stain image to the post-stain image (boundary + ICP + SIFT)
3. Estimates a scale factor via a Moffat fit on the post/pre ratio distribution
4. Subtracts the scaled pre-stain to reveal microbe-only signal, and produces diagnostic plots

The pipeline has **four stages** and saves checkpoints at each one, so stopping and re-running picks up where you left off.

---

## Setup

Run the cell below to install any missing dependencies, then the next cell to launch the GUI.

In [1]:
# Install dependencies if needed (only needs to run once)
# Uncomment the line below if you haven't installed these packages:
# !pip install numpy scipy opencv-python-headless matplotlib astropy tifffile Pillow ipywidgets

In [1]:
# Launch the interactive GUI
from exo2micro.gui import launch

gui = launch(
    output_dir='processed',   # where results are saved
    raw_dir='raw',        # where your raw images live
)

## How to use the GUI

### 1. Check your raw channels (recommended first step)
Click **Survey raw channels** to scan your raw TIFFs and confirm which RGB channel carries signal for each dye. This catches problems like wrong channel assignments before you run the full pipeline.

### 2. Enter your samples and dyes
Type one sample per line in the **Samples** box and one dye per line in the **Dyes** box. Or click **Auto-detect** to scan your raw image directory and populate both boxes.

### 3. Pick a scale method
The **Scale** dropdown controls how the background scale factor is estimated:
- **Auto (Moffat fit)** — the default. Fits a Moffat profile to the log-ratio distribution. Works well for most samples.
- **Auto + ratio percentile** — in addition to the Moffat fit, computes a scale from a user-chosen percentile (accepts decimal values like 99.1). Produces an extra difference image.
- **Auto + manual override** — in addition to the Moffat fit, uses a user-specified exact scale factor.
- **Auto + percentile + manual** — produces all three.

When multiple scale methods are active, the excess heatmap plot overlays all their scale lines side-by-side so you can compare visually.

### 4. Click ▶ Run Pipeline
The pipeline processes each sample × dye through all four stages. With **Show diagnostic plots inline** checked you'll see each stage's diagnostics as it finishes, with short captions explaining what to look for.

### 5. Inspect results
Use the **🔍 Zoom & Inspect** panel to zoom in on any region of the final difference image (or any stage output), with adjustable Gaussian blur. Check **Show post + aligned pre + diff side-by-side** to confirm suspicious features against the underlying data.

Use the **👁️ Blink Comparison** panel to visually verify alignment quality: load two alignment checkpoints and click the toggle button to flip between them.

### 6. (Optional) Adjust advanced parameters
Click the **Advanced Parameters** accordion to fine-tune alignment or diagnostic settings. Most users won't need to change anything here.

### 7. (Optional) Compare parameter values
Use the **Parameter Comparison** section to sweep one parameter across a list of values. For example, try comparing `boundary_width` at values 10, 15, 20 to see how it affects alignment quality on a tricky sample.

---

### Tips
- **Resume:** Checkpoints are saved after every stage. If you stop and re-run, the pipeline picks up where it left off.
- **Force rerun:** Check this to re-process from scratch instead of resuming.
- **From / To stage:** Pick a specific stage to start or stop at. Useful for re-running stage 4 with different scale settings without redoing alignment.
- **Parallel:** For large batches, check the parallel box and set the number of workers (start with 4).
- **Check Status:** Click this to see which stages have completed for each sample.

### Where are my results?
All output files are saved under the output directory you specified:
```
processed/
  {sample}/
    {dye}/
      tiff/              ← intermediate and final images (float32)
      fits/              ← same images with metadata headers
      pipeline_checks/   ← all diagnostic plots (heatmaps, histograms, ratio fit, difference image)
```

The main result is `tiff/04_difference_difference.tiff` (the Moffat-fit difference image). If you set `scale_percentile` or `manual_scale`, you'll also see additional files like `04_difference_difference_manual.tiff` and `04_difference_difference_percentile_p50.tiff`.

---

## Scripting API (for advanced users)

You can also use the pipeline programmatically without the GUI.

In [ ]:
# Process a single sample
import exo2micro as e2m

run = e2m.SampleDye('CD070', 'SybrGld_microbe')
run.set_params(boundary_width=20)  # optional: tweak a parameter
run.status()                       # see what's already been processed
# run.run()                        # uncomment to actually run

In [ ]:
# Run with additional scale methods alongside the default Moffat fit
# run = e2m.SampleDye('CD070', 'SybrGld_microbe')
# run.set_params(scale_percentile=50.0, manual_scale=1.2)
# run.run()
# This produces three difference images: Moffat, percentile-p50, and manual

In [ ]:
# Batch processing
# results = e2m.run_batch(
#     samples=['CD070', 'CD063', 'CD055'],
#     dyes=['SybrGld_microbe', 'DAPI'],
#     parallel=True,
#     n_workers=4,
# )

In [ ]:
# Zoom into a region of a difference image programmatically
# import tifffile
# from exo2micro import plot_zoom, plot_zoom_multi
#
# diff = tifffile.imread(
#     'processed/CD070/SybrGld_microbe/tiff/04_difference_difference.tiff')
# fig, crop = plot_zoom(
#     diff, row=5000, col=8000, size=800, sigma=2.0,
#     diverging=True,
#     title='CD070 / SybrGld_microbe - feature at (5000, 8000)',
#     save_path='feature_zoom.png')